# Seaborn Mini-Project: Full Risk Visualization Report
### Credit Card Risk Analysis Project

This is a capstone-style mini project pulling together everything from all 7 Seaborn
phases — themes, distributions, categorical comparisons, bivariate relationships,
correlation heatmaps, small multiples, and regression diagnostics — into one realistic
end-to-end deliverable.

**Format:** Each task has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. Tasks are less scaffolded than the phase notebooks — each one expects
you to combine skills, the way a real analysis would.

Run the setup cell first — it builds a fresh 700-applicant dataset with the full set of
features used across every phase, including deliberately redundant columns and a few
outliers.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(7)
sns.set_theme(style="whitegrid")

n = 700

age = np.clip(np.random.normal(40, 12, n), 18, 80)
housing_status = np.random.choice(['Rent', 'Own', 'Mortgage'], size=n, p=[0.35, 0.25, 0.40])
employment_type = np.random.choice(['Salaried', 'Self-Employed', 'Unemployed'], size=n, p=[0.6, 0.3, 0.1])
education_level = np.random.choice(['High School', 'Bachelor', 'Graduate'], size=n, p=[0.35, 0.45, 0.20])

annual_income = np.random.lognormal(mean=10.8, sigma=0.4, size=n)
# A few implausible high-income outliers, on purpose
outlier_idx = np.random.choice(n, size=5, replace=False)
annual_income[outlier_idx] = 5_000_000

loan_amount_requested = np.clip(annual_income * np.random.uniform(0.05, 0.30, size=n), 1000, None)
loan_amount_requested = np.where(housing_status == 'Rent',
                                  loan_amount_requested * 0.85, loan_amount_requested)

credit_score = np.clip(np.random.normal(660, 65, n), 300, 850)
# Deliberately redundant with Credit_Score
months_since_delinquency = np.clip(
    (credit_score - 300) / 550 * 60 + np.random.normal(0, 4, n), 0, 60
)

credit_limit = np.clip(3000 + annual_income * 0.15 + np.random.normal(0, 1500, n), 500, None)

total_debt = annual_income * np.random.uniform(0.05, 0.55, size=n)
debt_to_income = np.clip((total_debt / annual_income) * 100, 0, 90)
credit_utilization = np.clip(debt_to_income * 0.9 + np.random.normal(0, 5, n), 0, 100)

raw_risk = (debt_to_income / 100) * 0.6 + ((850 - credit_score) / 550) * 0.6
employment_bump = np.where(employment_type == 'Unemployed', 0.15, 0)
default_probability = np.clip(raw_risk + employment_bump + np.random.normal(0, 0.08, n), 0.01, 0.95)
default = np.random.binomial(1, default_probability)

risk_tier = pd.cut(credit_score, bins=[300, 600, 700, 850], labels=['High', 'Medium', 'Low'])
income_bracket = pd.cut(annual_income, bins=[0, 40000, 70000, 100000, np.inf],
                         labels=['<40k', '40k-70k', '70k-100k', '100k+'])

df = pd.DataFrame({
    'Age': age,
    'Housing_Status': housing_status,
    'Employment_Type': employment_type,
    'Education_Level': education_level,
    'Annual_Income': annual_income,
    'Loan_Amount_Requested': loan_amount_requested,
    'Credit_Score': credit_score,
    'Months_Since_Last_Delinquency': months_since_delinquency,
    'Credit_Limit': credit_limit,
    'Total_Debt': total_debt,
    'Debt_to_Income': debt_to_income,
    'Credit_Utilization': credit_utilization,
    'Default_Probability': default_probability,
    'Default': default,
    'Risk_Tier': risk_tier,
    'Income_Bracket': income_bracket
})

print(df.shape)
df.head()


---
## Task 1: Set the Report's Look, and Get Oriented

**Q1.** Set the project's overall visual theme with `sns.set_theme(style='whitegrid', palette='muted')`. Then plot a histogram with KDE of `Credit_Score` (`kde=True`), finished with a Matplotlib title — the classic Seaborn-draws / Matplotlib-finishes pattern.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')

sns.histplot(data=df, x='Credit_Score', kde=True)
plt.title("Credit Score Distribution")
plt.show()


---
## Task 2: Outlier Check

**Q2.** Plot a boxplot of `Annual_Income` — the $5,000,000 outliers should be obvious. Then plot it a second time with `plt.yscale('log')` so the box itself stays readable despite the outliers.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

sns.boxplot(data=df, y='Annual_Income', ax=axes[0])
axes[0].set_title("Linear Scale")

sns.boxplot(data=df, y='Annual_Income', ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title("Log Scale")

plt.show()


---
## Task 3: Risk Tier Density Profile

**Q3.** Plot a split violin plot of `Debt_to_Income` by `Risk_Tier` (ordered Low/Medium/High), with `hue='Default'` and `split=True`, `inner='quartile'`, and a `palette={0: 'seagreen', 1: 'tomato'}`.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
sns.violinplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Default',
               order=['Low', 'Medium', 'High'], split=True, inner='quartile',
               palette={0: 'seagreen', 1: 'tomato'})
plt.title("Debt-to-Income by Risk Tier, Split by Default")
plt.show()


---
## Task 4: Overall Default Rate

**Q4.** Plot `sns.countplot(data=df, x='Default')` with a custom `palette={0: 'seagreen', 1: 'tomato'}`, labeled with `ax.bar_label()`, and readable x-tick labels (`'Paid'` / `'Defaulted'`).

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(data=df, x='Default', hue='Default', palette={0: 'seagreen', 1: 'tomato'},
              legend=False, ax=ax)
ax.bar_label(ax.containers[0])
ax.set_xticks([0, 1])
ax.set_xticklabels(['Paid', 'Defaulted'])
ax.set_title("Total Applicants: Paid vs. Defaulted")
plt.show()


---
## Task 5: Loan Amount by Housing Status

**Q5.** Plot `sns.barplot(data=df, x='Housing_Status', y='Loan_Amount_Requested')`, sorted by descending average loan amount (`order=`), with a `bar_label()` formatted as currency.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
sorted_order = df.groupby('Housing_Status')['Loan_Amount_Requested'].mean().sort_values(ascending=False).index

ax = sns.barplot(data=df, x='Housing_Status', y='Loan_Amount_Requested', order=sorted_order)
ax.bar_label(ax.containers[0], fmt='$%.0f')
ax.set_title("Average Loan Amount Requested by Housing Status")
plt.show()


---
## Task 6: Income vs. Debt, by Default

**Q6.** Plot a scatterplot of `Annual_Income` vs. `Total_Debt`, colored by `Default` (custom palette), `alpha=0.5`, with a Matplotlib trend line (`np.polyfit`) overlaid. Restrict to applicants with `Annual_Income < 1_000_000` first (drop the extreme outliers) so the relationship among typical applicants isn't flattened.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
typical = df[df['Annual_Income'] < 1_000_000]

slope, intercept = np.polyfit(typical['Annual_Income'], typical['Total_Debt'], 1)
x_line = np.linspace(typical['Annual_Income'].min(), typical['Annual_Income'].max(), 100)
y_line = slope * x_line + intercept

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=typical, x='Annual_Income', y='Total_Debt', hue='Default',
                 palette={0: 'steelblue', 1: 'tomato'}, alpha=0.5, ax=ax)
ax.plot(x_line, y_line, color='black', linewidth=2, label='Trend')
ax.set_title("Income vs. Debt, by Default Status")
ax.legend()
plt.show()


---
## Task 7: Joint Relationship + Marginals

**Q7.** Plot `sns.jointplot()` of `Annual_Income` vs. `Total_Debt` (using the `typical` subset from Task 6), `kind='scatter'`, `hue='Default'`, custom palette, `alpha=0.6`. Add a suptitle via the returned `JointGrid`'s `.fig`.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
jg = sns.jointplot(data=typical, x='Annual_Income', y='Total_Debt', hue='Default',
                    palette={0: 'steelblue', 1: 'tomato'}, alpha=0.6, height=7)
jg.fig.suptitle("Income vs. Debt: Joint Distribution", y=1.02)
plt.show()


---
## Task 8: Multicollinearity Check

**Q8.** Build a correlation heatmap of `['Age', 'Annual_Income', 'Credit_Score', 'Months_Since_Last_Delinquency', 'Credit_Limit', 'Total_Debt', 'Debt_to_Income', 'Credit_Utilization', 'Default_Probability']`, masking the upper triangle, `annot=True`, `cmap='coolwarm'`, `center=0`. Then programmatically list every pair with `|correlation| > 0.7`.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
numeric_cols = ['Age', 'Annual_Income', 'Credit_Score', 'Months_Since_Last_Delinquency',
                'Credit_Limit', 'Total_Debt', 'Debt_to_Income', 'Credit_Utilization',
                'Default_Probability']
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.show()

corr_pairs = corr_matrix.unstack().reset_index()
corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
corr_pairs = corr_pairs[corr_pairs['Feature_1'] != corr_pairs['Feature_2']]
corr_pairs['pair_key'] = corr_pairs.apply(lambda r: tuple(sorted([r['Feature_1'], r['Feature_2']])), axis=1)
corr_pairs = corr_pairs.drop_duplicates(subset='pair_key').drop(columns='pair_key')
high_corr = corr_pairs[corr_pairs['Correlation'].abs() > 0.7].sort_values('Correlation', key=abs, ascending=False)
print(high_corr)


---
## Task 9: Small Multiples by Education

**Q9.** Use `sns.catplot(data=df, x='Default', y='Debt_to_Income', col='Education_Level', kind='box')` to facet the Default/DTI relationship by Education Level, then relabel titles/axes via the returned grid's `.set_titles()` and `.set_axis_labels()`.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
g = sns.catplot(data=df, x='Default', y='Debt_to_Income', col='Education_Level', kind='box')
g.set_titles("Education: {col_name}")
g.set_axis_labels("Default Status", "Debt-to-Income (%)")
plt.show()


---
## Task 10: Everything at Once

**Q10.** Plot `sns.pairplot()` on `df` using `vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income', 'Credit_Limit']` (restricted to `typical`, the outlier-free subset), `hue='Default'`, custom palette, `corner=True`, `diag_kind='kde'`.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
sns.pairplot(typical, vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income', 'Credit_Limit'],
             hue='Default', palette={0: 'seagreen', 1: 'tomato'}, corner=True, diag_kind='kde',
             plot_kws={'alpha': 0.5})
plt.show()


---
## Task 11: Does DTI Predict Default?

**Q11.** Plot `sns.regplot(data=df, x='Debt_to_Income', y='Default', logistic=True, scatter_kws={'alpha': 0.15})` — a direct visual preview of a logistic regression, showing estimated default probability as a function of DTI. (Requires the optional `statsmodels` package — if it's not installed, this will raise a `RuntimeError`; run `pip install statsmodels` first.)

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
try:
    sns.regplot(data=df, x='Debt_to_Income', y='Default', logistic=True, scatter_kws={'alpha': 0.15})
    plt.title("Probability of Default vs. Debt-to-Income")
    plt.show()
except RuntimeError as e:
    print(f"Skipped: {e}")


---
## Task 12: Every Applicant, Not Just Summaries

**Q12.** Plot `sns.violinplot()` of `Credit_Score` by `Risk_Tier` with `inner=None` in a light color, then overlay `sns.stripplot()` of the same data in black, `alpha=0.4`, `size=3`.

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.violinplot(data=df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'],
               inner=None, color='lightsteelblue', ax=ax)
sns.stripplot(data=df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'],
              color='black', alpha=0.4, size=3, ax=ax)
ax.set_title("Credit Score by Risk Tier: Density + Every Applicant")
plt.show()


---
## Task 13: A Precise Threshold Answer

**Q13.** Plot `sns.ecdfplot(data=df, x='Debt_to_Income')`, add reference lines at `x=40`, and print the exact percentage of applicants with `Debt_to_Income > 40`.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
pct_above_40 = (df['Debt_to_Income'] > 40).mean() * 100

fig, ax = plt.subplots(figsize=(9, 6))
sns.ecdfplot(data=df, x='Debt_to_Income', ax=ax)
ax.axvline(40, color='red', linestyle='--')
ax.set_title("Debt-to-Income: Empirical CDF")
plt.show()

print(f"{pct_above_40:.1f}% of applicants have Debt-to-Income above 40%")


---
## Task 14: The Default Rate Trend

**Q14.** Plot `sns.pointplot(data=df, x='Income_Bracket', y='Default', order=['<40k', '40k-70k', '70k-100k', '100k+'])` to show how default rate trends across income brackets, with `errorbar=None` for a clean line.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
bracket_order = ['<40k', '40k-70k', '70k-100k', '100k+']

sns.pointplot(data=df, x='Income_Bracket', y='Default', order=bracket_order,
              color='darkred', errorbar=None)
plt.title("Default Rate by Income Bracket")
plt.ylabel("Default Rate")
plt.show()


---
## Final Task: The Full Risk Report

**Q15.** Assemble a 2x2 dashboard Figure (`figsize=(15, 11)`):
- **Top-left:** the countplot of `Default` from Task 4
- **Top-right:** the correlation heatmap from Task 8 (just the matrix, no need to re-flag pairs)
- **Bottom-left:** the scatterplot of `Annual_Income` vs. `Total_Debt` colored by `Default`, from Task 6
- **Bottom-right:** the pointplot of default rate by `Income_Bracket` from Task 14

Add `fig.suptitle("Credit Card Risk Portfolio: Full Report")` in bold, call `fig.tight_layout()`, and save it with `fig.savefig('full_risk_report.png', dpi=150, bbox_inches='tight')`.

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Top-left: default counts
sns.countplot(data=df, x='Default', hue='Default', palette={0: 'seagreen', 1: 'tomato'},
              legend=False, ax=axes[0, 0])
axes[0, 0].bar_label(axes[0, 0].containers[0])
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_xticklabels(['Paid', 'Defaulted'])
axes[0, 0].set_title("Total Applicants: Paid vs. Defaulted")

# Top-right: correlation heatmap
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=axes[0, 1], cbar_kws={'shrink': 0.7})
axes[0, 1].set_title("Feature Correlation Matrix")

# Bottom-left: income vs debt
sns.scatterplot(data=typical, x='Annual_Income', y='Total_Debt', hue='Default',
                 palette={0: 'steelblue', 1: 'tomato'}, alpha=0.5, ax=axes[1, 0])
axes[1, 0].plot(x_line, y_line, color='black', linewidth=2)
axes[1, 0].set_title("Income vs. Debt, by Default Status")

# Bottom-right: default rate trend
sns.pointplot(data=df, x='Income_Bracket', y='Default', order=bracket_order,
              color='darkred', errorbar=None, ax=axes[1, 1])
axes[1, 1].set_title("Default Rate by Income Bracket")
axes[1, 1].set_ylabel("Default Rate")

fig.suptitle("Credit Card Risk Portfolio: Full Report", fontsize=17, fontweight='bold')
fig.tight_layout()
fig.savefig('full_risk_report.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved full_risk_report.png")


---
## Mini-Project Complete

You just built a report that touches every phase of the Seaborn curriculum: themes,
distributions (histplot/KDE, boxplot, violin), categorical comparisons (countplot,
barplot), bivariate relationships (scatterplot with hue, jointplot), a correlation
heatmap for multicollinearity, small multiples (catplot), a full pairplot, a logistic
regression preview, strip/violin overlays, an ECDF threshold answer, and a pointplot
trend — finishing in one saved, presentation-ready dashboard.

Between Matplotlib and Seaborn, this is the complete visualization toolkit for the
project. From here, the natural next step is applying all of it to your real dataset,
or moving into feature engineering and modeling.
